# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet
# Optional dependencies for visualization
!pip install matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata

print("Dataset Loaded:")
print(f"Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We use `dataset.record_sets` to enumerate record sets. For each, print its `@id` and `name` if available, then enumerate fields and columns by their `@id` and `name` as well.

In [ ]:
# List all record sets and their fields
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  RecordSet: {rs['@id']} (name: {rs.get('name', 'no name')})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for field in fields:
        print(f"      {field['@id']} (name: {field.get('name', 'no name')})")
    # If there are columns, print them
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("    Columns:")
        for col in columns:
            print(f"      {col['@id']} (name: {col.get('name', 'no name')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll use the `@id` of each record set and load all records as DataFrames keyed by their `@id`.

In [ ]:
# Gather @id for all record sets
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

for rs_id, df in dataframes.items():
    print(f"RecordSet {rs_id} columns:")
    print(df.columns.tolist())
    print(df.head())
    print("\n---\n")

## 4. Exploratory Data Analysis (EDA)
Perform basic data processing: filter records, normalize a numeric field, and group by a categorical field.

Choose the record set and field using their `@id`. Replace placeholder IDs below with actual dataset `@id`s.

In [ ]:
# You may need to identify the numeric and group fields from the overview above.
# Example record set and field IDs:
# record_set_id = 'cr:RecordSet/ordered-logistic-regression-results'
# numeric_field_id = 'cr:Field/log-likelihood'  # This is typical for logistic regression
# group_field_id = 'cr:Field/ward'

# For demonstration, we attempt to use the first available record set.
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Find a numeric field (e.g., log likelihood, coefficient, p-value)
    numeric_candidates = [c for c in df.columns if 'log' in c or 'coef' in c or 'pval' in c or 'likelihood' in c or 'std' in c]
    numeric_field = numeric_candidates[0] if numeric_candidates else df.select_dtypes(include=['float', 'int']).columns[0]

    # Filter records with the numeric field greater than a threshold
    threshold = df[numeric_field].mean() if 'mean' in dir(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Choose a categorical/group field (e.g., ward, county)
    cat_candidates = [c for c in df.columns if 'ward' in c or 'county' in c or 'group' in c]
    group_field = cat_candidates[0] if cat_candidates else df.select_dtypes(include=['object', 'category']).columns[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No record sets or records available to analyze.")

## 5. Visualization
Visualize numeric field distributions and relationship to categorical fields.

Below, we show a histogram and a boxplot (if data exists).

In [ ]:
if len(dataframes) > 0 and 'filtered_df' in locals():
    plt.figure(figsize=(10, 4))
    sns.histplot(filtered_df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No available data.")

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² dataset metadata and reviewed available record sets, fields, and columns using `mlcroissant`.
- Extracted records into DataFrames and identified key fields using their `@id`.
- Filtered, normalized, and grouped numeric data for exploratory analysis.
- Visualized distributions and relationships between key predictors and grouping fields.

This process demonstrates how Croissant-compatible datasets can be explored programmatically, ensuring traceability via consistent use of entity `@id` references.

You can further extend this analysis for model building, publication-quality plots, and policy insights using the FAIR² dataset.